# Single-cell RNA-seq of 3k PBMCs - Scanpy walkthrough

A standard single-cell RNA-seq analysis using the **scverse / Scanpy** ecosystem.
This notebook is my guided walkthrough for learning the workflow grammar before applying
it independently in `02_scrna_pbmc1k_yourturn.ipynb`.

**Data:** 3k PBMCs (10x Genomics), loaded automatically with Scanpy.

If a cell errors about `leidenalg` or `igraph`, run: `pip install leidenalg igraph`.

## 0. Setup

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

sc.settings.verbosity = 1                # 0=silent, 1=info
sc.settings.set_figure_params(dpi=80, facecolor='white')
print('scanpy', sc.__version__)

## 1. Load the data

`adata` is an **AnnData** object: a matrix of *cells x genes* plus tables of
per-cell metadata (`.obs`) and per-gene metadata (`.var`). This is the core data
structure of the whole scverse ecosystem.

In [ ]:
adata = sc.datasets.pbmc3k()   # raw UMI counts, ~2700 cells x ~32000 genes
adata.var_names_make_unique()
adata

## 2. Quality control

Each cell gets QC metrics: number of genes detected, total counts, and the fraction
of reads from **mitochondrial** genes (a high fraction usually means a dying cell).

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')   # human mito genes start with MT-
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe()

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

### Guided choice - filtering thresholds

Look at the plots above. The classic PBMC thresholds are below. Adjust them if your
plots suggest different cut-offs, and write one line saying why.

_My choice & reasoning:_ For this walkthrough I kept the classic PBMC3k thresholds:
`min_genes=200`, `n_genes_by_counts < 2500` and `pct_counts_mt < 5`. These remove
near-empty droplets, likely doublets with unusually many detected genes and cells with
a high mitochondrial fraction.

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)     # drop near-empty droplets
sc.pp.filter_genes(adata, min_cells=3)        # drop genes seen in <3 cells

adata = adata[adata.obs.n_genes_by_counts < 2500, :]   # drop likely doublets
adata = adata[adata.obs.pct_counts_mt < 5, :].copy()   # drop dying cells
print(adata.n_obs, 'cells remain')

## 3. Normalisation

Put every cell on the same total count (so sequencing depth doesn't dominate), then
log-transform. We keep a copy of the full log-normalised data in `.raw` for plotting later.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

## 4. Highly variable genes (HVGs)

Most genes are uninformative noise. We keep the genes that vary most across cells —
those carry the biological signal that separates cell types.

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)
adata = adata[:, adata.var.highly_variable].copy()

## 5. Scale + PCA

Scale each gene to unit variance, then run PCA to compress the data to a few dozen
informative components before building the cell graph.

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

## 6. Neighbourhood graph + UMAP

Connect each cell to its nearest neighbours in PCA space, then lay that graph out in 2D
with UMAP for visualisation.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)
sc.pl.umap(adata)

## 7. Clustering (Leiden)

The Leiden algorithm groups cells into communities on the neighbour graph. Higher
`resolution` = more, smaller clusters.

### Guided choice - clustering resolution
Try 0.5, 1.0 and 1.5. Which gives clusters that look clean on the UMAP and match the
number of PBMC cell types you expect? Note your choice.

_My choice & reasoning:_ I start with `resolution=1.0`, then compare lower and higher
values if the UMAP looks under- or over-split. For PBMC data, I expect several immune
lineages rather than one broad cluster, but I also want clusters that can be explained
by marker genes.

In [ ]:
sc.tl.leiden(adata, resolution=1.0)   # if this errors: pip install leidenalg igraph
sc.pl.umap(adata, color=['leiden'], legend_loc='on data')

## 8. Marker genes per cluster

For each cluster, rank the genes that are most specific to it. These are the clues we
use to name the cell types.

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False)

## 9. Cell-type annotation

Compare each cluster's top markers with these well-known PBMC genes:

| Cell type | Markers |
|---|---|
| CD4 T cells | IL7R, CD3D |
| CD8 T cells | CD8A, CD3D |
| B cells | MS4A1, CD79A |
| NK cells | GNLY, NKG7 |
| CD14+ monocytes | CD14, LYZ |
| FCGR3A+ monocytes | FCGR3A, MS4A7 |
| Dendritic cells | FCER1A, CST3 |
| Platelets | PPBP |

In [ ]:
markers = ['IL7R','CD3D','CD8A','MS4A1','CD79A','GNLY','NKG7',
           'CD14','LYZ','FCGR3A','MS4A7','FCER1A','CST3','PPBP']
sc.pl.dotplot(adata, markers, groupby='leiden')

### Guided choice - name the clusters

Using the dotplot, map each Leiden cluster number to a cell type. The exact labels can
change with Scanpy and Leiden versions, so I use the marker evidence rather than copying
a fixed answer. The independent notebook contains my completed annotation table.

In [ ]:
# Example mapping from the classic PBMC3k tutorial pattern.
# Re-check the dotplot before running this in a fresh environment.
example_names = ['CD4 T', 'CD14+ monocytes', 'B cells', 'CD8 T',
                 'NK cells', 'FCGR3A+ monocytes', 'dendritic cells', 'platelets']

if len(example_names) == adata.obs['leiden'].cat.categories.size:
    adata.rename_categories('leiden', example_names)
    adata.obs['cell_type'] = adata.obs['leiden']
    sc.pl.umap(adata, color='cell_type', legend_loc='on data')
else:
    print('Review the dotplot and create one label per cluster before renaming.')

## 10. Save & reflect

Save the processed object, and write 2-3 sentences: how many cells/clusters, which
types you found, and one thing that surprised you or that you'd do differently.

In [ ]:
import os
os.makedirs('results', exist_ok=True)
adata.write('results/pbmc3k_processed.h5ad')
print('saved -> results/pbmc3k_processed.h5ad')

**My reflection:** This walkthrough taught me the sequence of a basic Scanpy PBMC
analysis: QC, normalisation, HVG selection, PCA/neighbours/UMAP, Leiden clustering and
marker-based annotation. I used the same structure in notebook 2 on a different 10x
PBMC dataset and made the QC and annotation decisions from that dataset's plots.